# TALLER PRÁCTICO 3 - PREDICCIÓN METEOROLÓGICA MULTISTEP

## Seq2Seq con Bahdanau Attention aplicado a Jena Climate

**Grupo:** 3  
**Curso:** Deep Learning  
**Dataset:** Jena Climate (2009-2016)  
**Fuente:** [Kaggle - Jena Climate](https://www.kaggle.com/datasets/mnassrib/jena-climate)

---

### Objetivo general

Utilizar las **48 horas meteorológicas anteriores** para predecir la temperatura de las **24 horas siguientes** mediante un baseline diario y un modelo **Encoder-Decoder LSTM con Bahdanau Attention**.



### Edición para portafolio

Trabajo académico recuperado de la especialización. Se conservan el código, las explicaciones y las atribuciones originales. Se retiraron las salidas y los metadatos de ejecución para facilitar su lectura y revisión. Las conclusiones conservadas pertenecen a la entrega original; los entrenamientos de Deep Learning no se repitieron al organizar este repositorio. Ver [procedencia y autoría](../../docs/PROCEDENCIA.md).


## Preparación del entorno


In [ ]:
# Manejo de datos y visualización
import os
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Preprocesamiento y métricas
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

# Reproducibilidad y estilo visual
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
COLORES = {
    "azul": "#2563EB", "naranja": "#F97316",
    "verde": "#10B981", "morado": "#8B5CF6", "rojo": "#EF4444"
}
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#F8FAFC",
    "axes.titleweight": "bold", "axes.titlesize": 14,
    "axes.labelsize": 11, "legend.frameon": True,
})

dispositivos_gpu = tf.config.list_physical_devices("GPU")
print(f"TensorFlow: {tf.__version__}")
print("GPU detectada:" if dispositivos_gpu else "GPU no detectada; la ejecución en CPU sigue siendo válida.", dispositivos_gpu)


## Carga reproducible del dataset

Se utiliza el archivo ZIP o CSV proporcionado para la actividad. La celda busca primero un archivo ya cargado y, si no lo encuentra, abre el selector de archivos de Colab. Esto evita errores con credenciales de Kaggle.


In [ ]:
DATASET_URL = "https://www.kaggle.com/datasets/mnassrib/jena-climate"
NOMBRES_ESPERADOS = {
    "jena_climate_2009_2016.csv",
    "jena_climate_2009_2016.csv.zip",
    "jena-climate.zip",
}

def encontrar_archivo_dataset():
    carpetas = [
        Path.cwd(), Path("/content"), Path("/content/jena_climate"),
        Path("/content/jena_climate_data"), Path("jena_climate_data"),
    ]
    candidatos = []
    for carpeta in carpetas:
        if carpeta.exists():
            candidatos.extend(carpeta.glob("jena_climate_2009_2016.csv"))
            candidatos.extend(carpeta.glob("*jena*climate*.zip"))
    if candidatos:
        return candidatos[0]

    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "No se encontró el dataset. En Colab, suba el ZIP o el CSV; "
            "en Jupyter, cópielo en la misma carpeta del notebook."
        ) from exc

    print("Suba jena_climate_2009_2016.csv.zip o el CSV descomprimido:")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No se subió ningún archivo.")
    return Path(next(iter(uploaded)))

archivo_dataset = encontrar_archivo_dataset()
if archivo_dataset.suffix.lower() == ".zip":
    carpeta_datos = Path("/content/jena_climate_data") if Path("/content").exists() else Path("jena_climate_data")
    carpeta_datos.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archivo_dataset) as zf:
        csv_members = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        if len(csv_members) != 1:
            raise ValueError("El ZIP debe contener exactamente un archivo CSV.")
        zf.extract(csv_members[0], carpeta_datos)
        csv_path = carpeta_datos / csv_members[0]
elif archivo_dataset.suffix.lower() == ".csv":
    csv_path = archivo_dataset
else:
    raise ValueError("Formato no admitido. Use un archivo .zip o .csv.")

print("Dataset localizado correctamente:", csv_path)
print("Fuente:", DATASET_URL)


In [ ]:
print("Cargando archivo:", csv_path)
df = pd.read_csv(csv_path)
print("Dimensiones del dataset:", df.shape)
display(df.head())


# PARTE 1. Exploración de los datos

- Se revisaron dimensiones, columnas y tipos de datos.
- Se buscaron valores NaN y el centinela -9999 usado en las variables de viento.
- Se calcularon estadísticos descriptivos y el rango temporal.
- Se visualizaron temperatura, presión y humedad, cumpliendo el requisito de graficar al menos dos variables adicionales.



## 1.1 Dimensiones, variables y tipos


In [ ]:
# Dimensiones del dataset
print("Dimensiones (filas, columnas):", df.shape)
print()

# Nombres de las variables
print("Nombres de las columnas:")
print(df.columns.tolist())
print()

# Tipos de datos
print("Tipos de datos:")
print(df.dtypes)

## 1.2 Calidad de los datos: NaN y valores centinela


In [ ]:
faltantes_nan = df.isna().sum()
centinelas = df.select_dtypes(include=np.number).eq(-9999.0).sum()
calidad_datos = pd.DataFrame({"NaN": faltantes_nan, "Centinela -9999": centinelas})

print("Total de valores NaN:", int(faltantes_nan.sum()))
print("Centinelas en wv (m/s):", int(centinelas.get("wv (m/s)", 0)))
print("Centinelas en max. wv (m/s):", int(centinelas.get("max. wv (m/s)", 0)))


In [ ]:
ax = calidad_datos.loc[calidad_datos.sum(axis=1) > 0].plot(
    kind="bar", figsize=(8, 8), color=[COLORES["azul"], COLORES["naranja"]], width=0.72
)
ax.set_title("Calidad inicial: faltantes y valores centinela")
ax.set_xlabel("Variable")
ax.set_ylabel("Cantidad de registros")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 1.3 Estadísticos descriptivos


In [ ]:
df.describe().T

## 1.4 Fecha, rango temporal y orden cronológico


In [ ]:
date_col = "Date Time"
df[date_col] = pd.to_datetime(
    df[date_col], format="%d.%m.%Y %H:%M:%S", errors="raise"
)
df = df.set_index(date_col).sort_index()

print("Fecha inicial:", df.index.min())
print("Fecha final:  ", df.index.max())
print("Número total de registros:", len(df))
print("Marcas temporales duplicadas:", int(df.index.duplicated().sum()))


Las marcas temporales repetidas no se eliminan arbitrariamente: al convertir la serie a frecuencia horaria se consolidan mediante la función de agregación definida para cada variable.


## 1.5 Evolución de la temperatura


In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(df.index, df["T (degC)"], linewidth=0.45, color=COLORES["rojo"])
plt.title("Temperatura del aire - Serie completa (2009-2016)")
plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.tight_layout()
plt.show()


## 1.6 Otras variables meteorológicas


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 9), sharex=True)
axes[0].plot(df.index, df["p (mbar)"], color=COLORES["azul"], linewidth=0.45)
axes[0].set_title("Presión atmosférica")
axes[0].set_ylabel("p (mbar)")
axes[1].plot(df.index, df["rh (%)"], color=COLORES["verde"], linewidth=0.45)
axes[1].set_title("Humedad relativa")
axes[1].set_xlabel("Fecha")
axes[1].set_ylabel("rh (%)")
plt.tight_layout()
plt.show()



El conjunto contiene **420.551 registros y 15 variables**, desde enero de 2009 hasta enero de 2017. No hay NaN originales, pero sí 18 y 20 valores centinela -9999 en las variables de viento, que deben tratarse antes del modelado. Las gráficas evidencian variación diaria y estacional.

> En esta primera parte comprobamos que la información es multivariada y cronológica. El hallazgo de -9999 es importante porque no representa viento real, sino mediciones inválidas; por eso se corrige en la siguiente parte.


# PARTE 2. Conversión a datos horarios

- Se seleccionaron temperatura, presión, humedad y variables de viento.
- Se reemplazó -9999 por NaN antes de agregar.
- Temperatura, presión, humedad y velocidad media se promediaron; la velocidad máxima conservó el máximo horario.
- La dirección del viento se convirtió primero a seno y coseno para evitar promedios angulares incorrectos.


## 2.1 Variables seleccionadas


In [ ]:
columnas_interes = [
    "T (degC)", "p (mbar)", "rh (%)", "wv (m/s)",
    "max. wv (m/s)", "wd (deg)"
]
faltantes_columnas = [c for c in columnas_interes if c not in df.columns]
if faltantes_columnas:
    raise KeyError(f"Faltan columnas requeridas: {faltantes_columnas}")
df_sel = df[columnas_interes].copy()
print("Columnas seleccionadas:", columnas_interes)
display(df_sel.head())


## 2.2 Limpieza de los valores -9999


In [ ]:
# El dataset Jena Climate tiene un valor centinela -9999 en wv (m/s) y max. wv (m/s)
# que representa mediciones erróneas. Los reemplazamos por NaN antes de agregar.

for col in ["wv (m/s)", "max. wv (m/s)"]:
    if col in df_sel.columns:
        n_outliers = (df_sel[col] == -9999.0).sum()
        print(f"{col}: {n_outliers} valores -9999 detectados")
        df_sel.loc[df_sel[col] == -9999.0, col] = np.nan

## 2.3 Agregación de 10 minutos a 1 hora

| Variable | Agregación | Justificación |
|---|---:|---|
| T, p, rh, wv | Media | Resume el comportamiento típico de la hora. |
| max. wv | Máximo | Conserva el pico de viento observado. |
| wd | Media circular | Se promedian sus componentes seno y coseno, no los grados directamente. |


In [ ]:
# Separamos wd (deg) para tratarla como variable circular
if "wd (deg)" in df_sel.columns:
    wd_rad = np.deg2rad(df_sel["wd (deg)"])
    df_sel["wd_sin"] = np.sin(wd_rad)
    df_sel["wd_cos"] = np.cos(wd_rad)
    df_sel = df_sel.drop(columns=["wd (deg)"])

# Diccionario de agregación por variable
agg_dict = {}
for col in df_sel.columns:
    if col == "max. wv (m/s)":
        agg_dict[col] = "max"
    else:
        agg_dict[col] = "mean"

df_hourly = df_sel.resample("1h").agg(agg_dict)

print("Dimensiones tras la conversión horaria:", df_hourly.shape)
df_hourly.head()

## 2.4 Verificación de forma y faltantes horarios


In [ ]:
print("Shape horario:", df_hourly.shape)
print("\nValores faltantes antes de interpolar:")
print(df_hourly.isna().sum())
porcentaje_faltante = 100 * df_hourly["T (degC)"].isna().mean()
print(f"\nPorcentaje de horas sin temperatura: {porcentaje_faltante:.3f}%")


## 2.5 Tratamiento de los huecos horarios

Los huecos representan apenas **0,125 %** de la grilla. Se aplica interpolación temporal para mantener secuencias regulares de 48 y 24 horas. Como limitación metodológica, se reconoce que existen dos intervalos largos; por eso no se interpretan individualmente predicciones ubicadas dentro de esos periodos.


In [ ]:
# La grilla horaria necesita valores continuos para formar ventanas de longitud fija.
# El porcentaje faltante es 0.125%; se usa interpolación temporal y se documenta
# que las predicciones cercanas a los dos intervalos largos deben interpretarse con cautela.
df_hourly = df_hourly.interpolate(method="time", limit_area="inside")

faltantes_finales = int(df_hourly.isna().sum().sum())
if faltantes_finales != 0:
    raise ValueError(f"Quedaron {faltantes_finales} valores faltantes después de interpolar.")
print("Valores faltantes después de interpolar:", faltantes_finales)


## 2.6 Resultado visual: dos semanas a frecuencia horaria


In [ ]:
dos_semanas = df_hourly["T (degC)"].iloc[:24 * 14]
plt.figure(figsize=(9, 8))
plt.plot(dos_semanas.index, dos_semanas.values, color=COLORES["rojo"], linewidth=1.5)
plt.fill_between(dos_semanas.index, dos_semanas.values, alpha=0.12, color=COLORES["rojo"])
plt.title("Temperatura horaria - Primeras dos semanas")
plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.tight_layout()
plt.show()



La serie quedó regularizada en **70.129 horas y 7 variables**. El promedio es apropiado para variables continuas, el máximo conserva ráfagas y la dirección del viento se trató circularmente. La gráfica de dos semanas conserva la dinámica meteorológica y el ciclo diario.
 Aquí pasamos de mediciones cada diez minutos a una observación por hora. No aplicamos la misma operación a todo: usamos media, máximo y promedio circular según el significado físico de cada variable.


# PARTE 3. Ingeniería de características

- Se codificó la hora con hour_sin y hour_cos.
- Se añadieron year_sin y year_cos para representar la estacionalidad anual.
- Se verificó visualmente que la hora 23 y la hora 0 quedan próximas en el círculo.


## 3.1 Codificación cíclica de la hora


In [ ]:
# Extraemos la hora del día (0-23) a partir del índice temporal
horas = df_hourly.index.hour

# Codificación cíclica mediante seno y coseno
df_hourly["hour_sin"] = np.sin(2 * np.pi * horas / 24)
df_hourly["hour_cos"] = np.cos(2 * np.pi * horas / 24)

df_hourly[["hour_sin", "hour_cos"]].head(10)

In [ ]:
un_dia = df_hourly.iloc[:24]
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(un_dia["hour_sin"], un_dia["hour_cos"], marker="o", color=COLORES["morado"], lw=2)
for i, hora in enumerate(range(24)):
    ax.annotate(str(hora), (un_dia["hour_sin"].iloc[i], un_dia["hour_cos"].iloc[i]), fontsize=9)
ax.set_title("Codificación cíclica de la hora")
ax.set_xlabel("hour_sin")
ax.set_ylabel("hour_cos")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


### Respuesta taller

Representar directamente las horas con valores de 0 a 23 genera una distancia artificialmente grande entre 23:00 y 00:00, aunque en realidad son horas consecutivas. La pareja seno-coseno ubica las horas sobre un círculo: preserva su orden y hace que el final y el comienzo del día queden próximos.


## 3.2 Estacionalidad anual adicional


In [ ]:
dia_del_anio = df_hourly.index.dayofyear
dias_en_anio = 365.25

df_hourly["year_sin"] = np.sin(2 * np.pi * dia_del_anio / dias_en_anio)
df_hourly["year_cos"] = np.cos(2 * np.pi * dia_del_anio / dias_en_anio)

df_hourly[["year_sin", "year_cos"]].head()

## 3.3 Variables finales


In [ ]:
print("Columnas finales:", df_hourly.columns.tolist())
print("Dimensiones:", df_hourly.shape)
df_hourly.head()

El modelo recibe **11 características**: cinco mediciones meteorológicas, dos componentes de dirección del viento, dos componentes de hora y dos componentes anuales. La codificación circular evita discontinuidades artificiales y ayuda a representar ciclos diarios y estacionales.

La idea clave es que el tiempo es cíclico. Con seno y coseno, las 23:00 y las 00:00 quedan cerca, algo que no ocurre si usamos simplemente los números 23 y 0.


# PARTE 4. División cronológica de los datos

- Se asignó 70 % al entrenamiento, 15 % a validación y 15 % a prueba.
- No se barajaron las observaciones.
- Se mostraron los rangos de fechas de cada subconjunto.



## 4.1 División 70 % / 15 % / 15 %


In [ ]:
n = len(df_hourly)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)  # 70% + 15%

df_train = df_hourly.iloc[:train_end]
df_val   = df_hourly.iloc[train_end:val_end]
df_test  = df_hourly.iloc[val_end:]

print("Tamaño total:", n)
print(f"Train: {len(df_train)} registros ({len(df_train)/n*100:.1f}%)")
print(f"Val:   {len(df_val)} registros ({len(df_val)/n*100:.1f}%)")
print(f"Test:  {len(df_test)} registros ({len(df_test)/n*100:.1f}%)")

## 4.2 Rangos temporales


In [ ]:
print("Train:", df_train.index.min(), "->", df_train.index.max())
print("Val:  ", df_val.index.min(), "->", df_val.index.max())
print("Test: ", df_test.index.min(), "->", df_test.index.max())


## 4.3 Resultado visual de la división


In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(df_train.index, df_train["T (degC)"], label="Entrenamiento", color=COLORES["azul"], linewidth=0.5)
plt.plot(df_val.index, df_val["T (degC)"], label="Validación", color=COLORES["naranja"], linewidth=0.5)
plt.plot(df_test.index, df_test["T (degC)"], label="Prueba", color=COLORES["verde"], linewidth=0.5)
plt.title("División cronológica del dataset")
plt.xlabel("Fecha")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.tight_layout()
plt.show()


La división conserva la secuencia temporal: **49.090 horas para entrenamiento, 10.519 para validación y 10.520 para prueba**. De esta manera, el rendimiento final se evalúa exclusivamente sobre el periodo más reciente.




# PARTE 5. Normalización

- Se definió la temperatura como objetivo.
- Se ajustaron dos StandardScaler únicamente con entrenamiento: uno para X y otro para Y.
- Los mismos parámetros se aplicaron a validación y prueba.


## 5.1 Variables de entrada y objetivo


In [ ]:
# Variable objetivo: Temperatura
target_col = "T (degC)"

# Todas las columnas del dataframe se usarán como features de entrada,
# incluyendo T (degC) ya que el modelo también usa la temperatura pasada
# como parte del contexto de entrada.
feature_cols = df_hourly.columns.tolist()

print("Variables de entrada (features):", feature_cols)
print("Variable objetivo (target):", target_col)
print("Número total de features:", len(feature_cols))

## 5.2 Ajuste de los escaladores solamente con entrenamiento


In [ ]:
# Usamos StandardScaler (media 0, desviación 1), adecuado para variables
# meteorológicas con distribuciones aproximadamente normales.
scaler_X = StandardScaler()
scaler_X.fit(df_train[feature_cols])

# Escalador separado para el target, útil luego para invertir la normalización
# de las predicciones y obtener el error en unidades reales (°C)
scaler_y = StandardScaler()
scaler_y.fit(df_train[[target_col]])

print("Media (features) aprendida del set de entrenamiento:")
print(pd.Series(scaler_X.mean_, index=feature_cols))

## 5.3 Transformación de los tres subconjuntos


In [ ]:
# Transformamos los features
X_train_scaled = scaler_X.transform(df_train[feature_cols])
X_val_scaled   = scaler_X.transform(df_val[feature_cols])
X_test_scaled  = scaler_X.transform(df_test[feature_cols])

# Transformamos el target
y_train_scaled = scaler_y.transform(df_train[[target_col]])
y_val_scaled   = scaler_y.transform(df_val[[target_col]])
y_test_scaled  = scaler_y.transform(df_test[[target_col]])

# Reconstruimos DataFrames para mantener el índice temporal (útil más adelante)
X_train_df = pd.DataFrame(X_train_scaled, columns=feature_cols, index=df_train.index)
X_val_df   = pd.DataFrame(X_val_scaled, columns=feature_cols, index=df_val.index)
X_test_df  = pd.DataFrame(X_test_scaled, columns=feature_cols, index=df_test.index)

y_train_df = pd.DataFrame(y_train_scaled, columns=[target_col], index=df_train.index)
y_val_df   = pd.DataFrame(y_val_scaled, columns=[target_col], index=df_val.index)
y_test_df  = pd.DataFrame(y_test_scaled, columns=[target_col], index=df_test.index)

print("X_train_df shape:", X_train_df.shape)
print("X_val_df shape:  ", X_val_df.shape)
print("X_test_df shape: ", X_test_df.shape)

## 5.4 Verificación numérica


In [ ]:
print("Estadísticos de X_train_df (deben tener media ≈ 0, std ≈ 1):")
print(X_train_df.describe().loc[["mean", "std"]].T)

print("\nEstadísticos de X_val_df (NO necesariamente 0 y 1, ya que se usó el scaler de train):")
print(X_val_df.describe().loc[["mean", "std"]].T)

## 5.5 Resultado visual de la normalización


In [ ]:
resumen_escalado = pd.DataFrame({
    "Media": X_train_df.mean(),
    "Desviación estándar": X_train_df.std(ddof=0),
})
fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
resumen_escalado["Media"].plot(kind="bar", ax=axes[0], color=COLORES["azul"])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Media después de normalizar")
resumen_escalado["Desviación estándar"].plot(kind="bar", ax=axes[1], color=COLORES["verde"])
axes[1].axhline(1, color="black", linewidth=1, linestyle="--")
axes[1].set_title("Desviación estándar después de normalizar")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


En entrenamiento, las variables quedan aproximadamente con **media 0 y desviación estándar 1**. Validación y prueba no tienen por qué presentar exactamente esos valores porque fueron transformadas con parámetros aprendidos exclusivamente del pasado.

Este procedimiento evita data leakage. Si ajustáramos el escalador con todo el dataset, el modelo conocería indirectamente la distribución del futuro y las métricas serían demasiado optimistas.


# PARTE 6. Construcción de ventanas

- Se construyeron ventanas deslizantes sin mezclar subconjuntos.
- Cada X tiene forma 48 x F y cada Y tiene forma 24 x 1.
- Se verificaron formas y una muestra visual.


## 6.1 Función de ventanas deslizantes


In [ ]:
def crear_ventanas(X_data, y_data, Tin=48, Tout=24):
    """
    Construye secuencias de entrada/salida para el modelo Seq2Seq.

    X_data: array (N, F) con las features ya normalizadas
    y_data: array (N, 1) con el target ya normalizado
    Tin:    horas de entrada (contexto pasado)
    Tout:   horas de salida (horizonte a predecir)

    Retorna:
    X: array (num_muestras, Tin, F)
    y: array (num_muestras, Tout, 1)
    """
    X, y = [], []
    n = len(X_data)

    for i in range(n - Tin - Tout + 1):
        X.append(X_data[i : i + Tin])
        y.append(y_data[i + Tin : i + Tin + Tout])

    return np.array(X), np.array(y)

## 6.2 Aplicación a entrenamiento, validación y prueba


In [ ]:
Tin = 48
Tout = 24

X_train, y_train = crear_ventanas(X_train_df.values, y_train_df.values, Tin, Tout)
X_val,   y_val   = crear_ventanas(X_val_df.values,   y_val_df.values,   Tin, Tout)
X_test,  y_test  = crear_ventanas(X_test_df.values,  y_test_df.values,  Tin, Tout)

print("X_train.shape:", X_train.shape)
print("y_train.shape:", y_train.shape)
print("X_val.shape:  ", X_val.shape)
print("y_val.shape:  ", y_val.shape)
print("X_test.shape: ", X_test.shape)
print("y_test.shape: ", y_test.shape)

## 6.3 Ejemplo visual de una ventana


In [ ]:
idx_muestra = 0
n_features = X_train.shape[2]

print(f"Ejemplo de ventana #{idx_muestra}")
print(f"Entrada (48 horas x {n_features} features): shape = {X_train[idx_muestra].shape}")
print(f"Salida (24 horas x 1 target):     shape = {y_train[idx_muestra].shape}")

fig, axes = plt.subplots(2, 1, figsize=(9, 9))

# Índice de la columna target dentro de feature_cols
idx_target_feature = feature_cols.index(target_col)

axes[0].plot(range(Tin), X_train[idx_muestra, :, idx_target_feature], marker="o", markersize=3)
axes[0].set_title("Entrada: Temperatura (48h pasadas, normalizada)")
axes[0].set_xlabel("Horas pasadas")
axes[0].set_ylabel("T normalizada")

axes[1].plot(range(Tout), y_train[idx_muestra, :, 0], marker="o", markersize=3, color="tab:red")
axes[1].set_title("Salida: Temperatura (24h futuras, normalizada)")
axes[1].set_xlabel("Horas futuras")
axes[1].set_ylabel("T normalizada")

plt.tight_layout()
plt.show()


## 6.4 Formas finales requeridas


In [ ]:
F = X_train.shape[2]
print(f"Número de features de entrada (F): {F}")
print(f"Tin (horas pasadas): {Tin}")
print(f"Tout (horas futuras): {Tout}")
print()
print("Dimensiones esperadas según el taller:")
print(f"  X ∈ R^(N x {Tin} x {F})  -> obtenido: {X_train.shape}")
print(f"  Y ∈ R^(N x {Tout} x 1)   -> obtenido: {y_train.shape}")


Las formas obtenidas cumplen el taller: **X = (N, 48, 11)** y **Y = (N, 24, 1)**. Cada muestra contiene dos días de contexto y el objetivo corresponde al día siguiente completo.

Cada ventana se puede leer así: entregamos al modelo 48 horas y once características por hora; a cambio esperamos 24 temperaturas, una para cada hora futura.


# PARTE 7. Modelo baseline de patrón diario
- Se copiaron las últimas 24 temperaturas de la ventana de entrada.
- Se desnormalizaron valores reales y predicciones.
- Se calcularon MAE, RMSE y R² sobre prueba.
- Se analizó cómo cambia el RMSE según el horizonte.


## 7.1 Construcción del baseline


In [ ]:
idx_target_feature = feature_cols.index(target_col)

def baseline_prediccion(X_data, idx_target_feature, Tout=24):
    """
    Extrae las últimas Tout horas de temperatura de la ventana de entrada,
    que corresponden al mismo horario del día anterior.
    """
    # X_data shape: (N, Tin, F) -> tomamos las últimas Tout horas del target
    return X_data[:, -Tout:, idx_target_feature:idx_target_feature+1]

y_pred_baseline_test = baseline_prediccion(X_test, idx_target_feature, Tout)

print("Shape predicción baseline:", y_pred_baseline_test.shape)
print("Shape y_test real:        ", y_test.shape)

## 7.2 Desnormalización


In [ ]:
def desnormalizar_target(y_scaled, scaler):
    """
    y_scaled: array (N, Tout, 1)
    Retorna: array (N, Tout) en unidades reales
    """
    n_samples, tout, _ = y_scaled.shape
    y_flat = y_scaled.reshape(-1, 1)
    y_inv = scaler.inverse_transform(y_flat)
    return y_inv.reshape(n_samples, tout)

y_test_real = desnormalizar_target(y_test, scaler_y)
y_pred_baseline_real = desnormalizar_target(y_pred_baseline_test, scaler_y)

print("y_test_real.shape:", y_test_real.shape)
print("y_pred_baseline_real.shape:", y_pred_baseline_real.shape)

## 7.3 Métricas del baseline


In [ ]:
def calcular_metricas(y_true, y_pred, nombre_modelo="Modelo"):
    """
    y_true, y_pred: arrays (N, Tout) en unidades reales
    Calcula MAE, RMSE y R2 sobre todos los horizontes juntos.
    """
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()

    mae  = mean_absolute_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    r2   = r2_score(y_true_flat, y_pred_flat)

    print(f"--- {nombre_modelo} ---")
    print(f"MAE:  {mae:.4f} °C")
    print(f"RMSE: {rmse:.4f} °C")
    print(f"R2:   {r2:.4f}")

    return {"modelo": nombre_modelo, "MAE": mae, "RMSE": rmse, "R2": r2}

metricas_baseline = calcular_metricas(y_test_real, y_pred_baseline_real, "Baseline (patrón diario)")

## 7.4 Predicción de ejemplo


In [ ]:
idx_muestra = 0

plt.figure(figsize=(8, 8))
plt.plot(range(Tout), y_test_real[idx_muestra], label="Real", marker="o")
plt.plot(range(Tout), y_pred_baseline_real[idx_muestra], label="Baseline (patrón diario)", marker="x")
plt.title("Predicción baseline vs. valor real (24 horas futuras)")
plt.xlabel("Horizonte (horas)")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.tight_layout()
plt.show()


## 7.5 Error por horizonte


In [ ]:
rmse_por_horizonte_baseline = []
for h in range(Tout):
    rmse_h = np.sqrt(mean_squared_error(y_test_real[:, h], y_pred_baseline_real[:, h]))
    rmse_por_horizonte_baseline.append(rmse_h)

plt.figure(figsize=(8, 8))
plt.plot(range(1, Tout+1), rmse_por_horizonte_baseline, marker="o")
plt.title("RMSE por horizonte - Baseline (patrón diario)")
plt.xlabel("Horizonte (horas hacia el futuro)")
plt.ylabel("RMSE (°C)")
plt.tight_layout()
plt.show()


El baseline obtiene **MAE = 2,4841 °C, RMSE = 3,2318 °C y R² = 0,8276**. Es una referencia fuerte porque aprovecha la periodicidad diaria, pero no puede adaptarse a cambios meteorológicos repentinos.

El baseline responde a la pregunta: qué pasaría si mañana repitiera el patrón térmico de hoy. Nuestro modelo neuronal debe mejorar claramente estas métricas para justificar su complejidad.


# PARTE 8. Modelo Seq2Seq con Bahdanau Attention

- El Encoder LSTM resume la secuencia y conserva todos sus estados ocultos.
- Bahdanau Attention calcula una combinación ponderada de esos estados.
- El Decoder LSTM genera una temperatura por paso.
- Durante entrenamiento se usa teacher forcing; durante prueba, inferencia autoregresiva.
- EarlyStopping restaura la mejor época y ModelCheckpoint guarda sus pesos.



## 8.1 Flujo conceptual

**Entrada (48 x 11) -> Encoder LSTM -> estados ocultos H -> Bahdanau Attention -> Decoder LSTM -> salida (24 x 1)**

- **Encoder:** procesa las 48 horas.
- **Estados ocultos:** guardan representaciones temporales del pasado.
- **Attention:** asigna pesos a las 48 posiciones.
- **Decoder:** genera una salida por horizonte.
- **Capa Dense:** convierte cada estado del decoder en una temperatura normalizada.


## 8.2 Entrada del decoder y teacher forcing


In [ ]:
def preparar_decoder_input(X_data, y_data, idx_target_feature):
    """
    Construye la secuencia de entrada del decoder desplazada (teacher forcing).
    decoder_input[t] = y_real[t-1], y decoder_input[0] = última T observada en X.
    """
    N, Tout, _ = y_data.shape
    ultima_T_entrada = X_data[:, -1, idx_target_feature:idx_target_feature+1]  # (N, 1)

    decoder_input = np.zeros_like(y_data)  # (N, Tout, 1)
    decoder_input[:, 0, :] = ultima_T_entrada[:, 0:1]
    decoder_input[:, 1:, :] = y_data[:, :-1, :]

    return decoder_input

dec_input_train = preparar_decoder_input(X_train, y_train, idx_target_feature)
dec_input_val   = preparar_decoder_input(X_val, y_val, idx_target_feature)
dec_input_test  = preparar_decoder_input(X_test, y_test, idx_target_feature)

print("dec_input_train.shape:", dec_input_train.shape)

## 8.3 Mecanismo de Bahdanau Attention


In [ ]:
class BahdanauAttention(layers.Layer):
    """
    Mecanismo de atención tipo Bahdanau (additive attention).

    Recibe:
      - query: estado oculto actual del decoder, shape (batch, hidden_dim)
      - values: salidas del encoder (todos los estados ocultos), shape (batch, Tin, hidden_dim)

    Retorna:
      - context_vector: vector de contexto ponderado, shape (batch, hidden_dim)
      - attention_weights: pesos de atención, shape (batch, Tin, 1)
    """
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.W1 = layers.Dense(units)  # proyecta el query (decoder)
        self.W2 = layers.Dense(units)  # proyecta los values (encoder)
        self.V  = layers.Dense(1)      # combina en un score escalar

    def call(self, query, values):
        # query: (batch, hidden_dim) -> (batch, 1, hidden_dim) para poder sumar con values
        query_expanded = tf.expand_dims(query, 1)

        # score: (batch, Tin, 1)
        score = self.V(tf.nn.tanh(self.W1(query_expanded) + self.W2(values)))

        # attention_weights: (batch, Tin, 1) -> normalizados con softmax sobre el eje temporal
        attention_weights = tf.nn.softmax(score, axis=1)

        # context_vector: suma ponderada de los values -> (batch, hidden_dim)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

## 8.4 Hiperparámetros


In [ ]:
# dimensión de los estados ocultos del encoder/decoder
HIDDEN_DIM = 64
# dimensión interna de la capa de atención
ATTENTION_DIM = 32

# número de features de entrada
F = X_train.shape[2]
Tin = X_train.shape[1]
Tout = y_train.shape[1]

print(f"F={F}, Tin={Tin}, Tout={Tout}, HIDDEN_DIM={HIDDEN_DIM}")

## 8.5 Encoder LSTM


In [ ]:
# --- Entradas ---
encoder_inputs = layers.Input(shape=(Tin, F), name="encoder_input")

# --- Encoder LSTM ---
# return_sequences=True -> necesitamos TODOS los estados ocultos (H) para la atención
# return_state=True     -> necesitamos el último estado (h, c) para inicializar el decoder
encoder_lstm = layers.LSTM(
    HIDDEN_DIM,
    return_sequences=True,
    return_state=True,
    name="encoder_lstm"
)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)

print("encoder_outputs (H):", encoder_outputs.shape)  # (batch, Tin, HIDDEN_DIM)
print("state_h:", state_h.shape)
print("state_c:", state_c.shape)

## 8.6 Decoder LSTM con Attention


In [ ]:
# --- Entrada del decoder (teacher forcing) ---
decoder_inputs = layers.Input(shape=(Tout, 1), name="decoder_input")

decoder_lstm_cell = layers.LSTMCell(HIDDEN_DIM, name="decoder_lstm_cell")
attention_layer = BahdanauAttention(ATTENTION_DIM, name="bahdanau_attention")
output_dense = layers.Dense(1, name="output_layer")  # capa de salida: predicción de T

# Estado inicial del decoder = último estado del encoder
dec_state_h, dec_state_c = state_h, state_c

all_outputs = []
all_attention_weights = []

for t in range(Tout):
    # Entrada del decoder en el paso t (teacher forcing): (batch, 1)
    dec_input_t = decoder_inputs[:, t, :]

    # 1) Calculamos el contexto de atención usando el estado oculto actual del decoder
    context_vector, attn_weights = attention_layer(dec_state_h, encoder_outputs)

    # 2) Concatenamos la entrada real del paso anterior con el contexto de atención
    lstm_input = layers.Concatenate(axis=-1)([dec_input_t, context_vector])

    # 3) Un paso de la LSTMCell del decoder
    dec_output, [dec_state_h, dec_state_c] = decoder_lstm_cell(
        lstm_input, states=[dec_state_h, dec_state_c]
    )

    # 4) Capa de salida: combinamos salida del decoder + contexto -> predicción escalar
    combined = layers.Concatenate(axis=-1)([dec_output, context_vector])
    prediction_t = output_dense(combined)  # (batch, 1)

    all_outputs.append(prediction_t)
    all_attention_weights.append(attn_weights)  # (batch, Tin, 1)

# Apilamos las salidas de todos los pasos: (batch, Tout, 1)
decoder_outputs = layers.Lambda(lambda x: tf.stack(x, axis=1), name="stack_outputs")(all_outputs)

# Apilamos los pesos de atención: (batch, Tout, Tin, 1) -> luego (batch, Tout, Tin)
attention_matrix = layers.Lambda(
    lambda x: tf.squeeze(tf.stack(x, axis=1), axis=-1), name="stack_attention"
)(all_attention_weights)

print("decoder_outputs shape:", decoder_outputs.shape)
print("attention_matrix shape:", attention_matrix.shape)

## 8.7 Modelo de entrenamiento


In [ ]:
model_seq2seq = Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=decoder_outputs,
    name="Seq2Seq_Attention"
)

model_seq2seq.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model_seq2seq.summary()

## 8.8 Modelo auxiliar para exponer la matriz de Attention


In [ ]:
model_seq2seq_with_attention = Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs=[decoder_outputs, attention_matrix],
    name="Seq2Seq_Attention_debug"
)
model_seq2seq_with_attention.summary()

## 8.9 EarlyStopping y guardado de los mejores pesos


In [ ]:
os.makedirs("/content/modelos", exist_ok=True)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath="/content/modelos/mejor_seq2seq_attention.weights.h5",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,   # guardamos solo pesos, por el Lambda/loop custom
    verbose=1
)

callbacks_list = [early_stopping, checkpoint]

## 8.10 Entrenamiento


In [ ]:
start_time = time.time()

history = model_seq2seq.fit(
    x=[X_train, dec_input_train],
    y=y_train,
    validation_data=([X_val, dec_input_val], y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks_list,
    verbose=1
)

training_time_seq2seq = time.time() - start_time
print(f"\nTiempo total de entrenamiento: {training_time_seq2seq:.2f} segundos "
      f"({training_time_seq2seq/60:.2f} minutos)")

## 8.11 Curvas de entrenamiento


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 9))
axes[0].plot(history.history["loss"], label="Entrenamiento", color=COLORES["azul"], marker="o")
axes[0].plot(history.history["val_loss"], label="Validación", color=COLORES["naranja"], marker="o")
axes[0].set_title("Pérdida MSE durante el entrenamiento")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[1].plot(history.history["mae"], label="Entrenamiento", color=COLORES["azul"], marker="o")
axes[1].plot(history.history["val_mae"], label="Validación", color=COLORES["naranja"], marker="o")
axes[1].set_title("MAE normalizado durante el entrenamiento")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("MAE")
axes[1].legend()
plt.tight_layout()
plt.show()


## 8.12 Mejor época y ruta del modelo


In [ ]:
best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
best_val_loss = float(np.min(history.history["val_loss"]))
RUTA_MEJORES_PESOS = "/content/modelos/mejor_seq2seq_attention.weights.h5"
print(f"Mejor época: {best_epoch}")
print(f"Mejor val_loss: {best_val_loss:.6f}")
print("Pesos del mejor modelo guardados en:", RUTA_MEJORES_PESOS)
n_params_seq2seq = model_seq2seq.count_params()
print(f"Número de parámetros del modelo: {n_params_seq2seq:,}")


## 8.13 Inferencia autoregresiva


In [ ]:
inf_encoder_inputs = layers.Input(shape=(Tin, F), name="inf_encoder_input")
inf_first_input = layers.Input(shape=(1,), name="inf_first_decoder_input")  # última T conocida

# --- Encoder (pesos entrenados) ---
inf_encoder_outputs, inf_state_h, inf_state_c = encoder_lstm(inf_encoder_inputs)

inf_dec_state_h, inf_dec_state_c = inf_state_h, inf_state_c
inf_dec_input_t = inf_first_input  # (batch, 1)

inf_all_outputs = []
inf_all_attention = []

for t in range(Tout):
    # Atención sobre el estado actual del decoder
    context_vector, attn_weights = attention_layer(inf_dec_state_h, inf_encoder_outputs)

    # Entrada al LSTMCell: predicción anterior (o T inicial) + contexto
    lstm_input = layers.Concatenate(axis=-1)([inf_dec_input_t, context_vector])

    dec_output, [inf_dec_state_h, inf_dec_state_c] = decoder_lstm_cell(
        lstm_input, states=[inf_dec_state_h, inf_dec_state_c]
    )

    combined = layers.Concatenate(axis=-1)([dec_output, context_vector])
    prediction_t = output_dense(combined)  # (batch, 1)

    inf_all_outputs.append(prediction_t)
    inf_all_attention.append(attn_weights)

    # Retroalimentación: la predicción actual se usa como entrada del siguiente paso
    inf_dec_input_t = prediction_t

inf_decoder_outputs = layers.Lambda(lambda x: tf.stack(x, axis=1), name="inf_stack_outputs")(inf_all_outputs)
inf_attention_matrix = layers.Lambda(
    lambda x: tf.squeeze(tf.stack(x, axis=1), axis=-1), name="inf_stack_attention"
)(inf_all_attention)

model_inference = Model(
    inputs=[inf_encoder_inputs, inf_first_input],
    outputs=[inf_decoder_outputs, inf_attention_matrix],
    name="Seq2Seq_Attention_Inference"
)

print("Modelo de inferencia construido (comparte pesos con el modelo de entrenamiento)")
model_inference.summary()

In [ ]:
# Primera entrada del decoder en inferencia: última T observada en la ventana de 48h
primera_T_test = X_test[:, -1, idx_target_feature:idx_target_feature+1]  # (N, 1)

y_pred_test_scaled, attention_test = model_inference.predict(
    [X_test, primera_T_test], batch_size=64, verbose=1
)

print("y_pred_test_scaled.shape:", y_pred_test_scaled.shape)
print("attention_test.shape:    ", attention_test.shape)  # (N, Tout, Tin) -> (N, 24, 48)

## 8.14 Métricas finales en prueba


In [ ]:
y_pred_seq2seq_real = desnormalizar_target(y_pred_test_scaled, scaler_y)
# y_test_real ya lo calculamos en la Parte VII (celda 31)

metricas_seq2seq = calcular_metricas(y_test_real, y_pred_seq2seq_real, "Seq2Seq + Attention")

print("\n--- Comparación rápida ---")
print(f"Baseline  -> MAE: {metricas_baseline['MAE']:.4f} | RMSE: {metricas_baseline['RMSE']:.4f} | R2: {metricas_baseline['R2']:.4f}")
print(f"Seq2Seq   -> MAE: {metricas_seq2seq['MAE']:.4f} | RMSE: {metricas_seq2seq['RMSE']:.4f} | R2: {metricas_seq2seq['R2']:.4f}")

## 8.15 Comparación visual contra el baseline


In [ ]:
comparacion_metricas = pd.DataFrame({
    "Baseline": metricas_baseline,
    "Seq2Seq + Attention": metricas_seq2seq,
}).T
mejora_mae = 100 * (metricas_baseline["MAE"] - metricas_seq2seq["MAE"]) / metricas_baseline["MAE"]
mejora_rmse = 100 * (metricas_baseline["RMSE"] - metricas_seq2seq["RMSE"]) / metricas_baseline["RMSE"]

fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for ax, metrica in zip(axes.flat[:3], ["MAE", "RMSE", "R2"]):
    valores = [metricas_baseline[metrica], metricas_seq2seq[metrica]]
    barras = ax.bar(["Baseline", "Seq2Seq"], valores, color=[COLORES["naranja"], COLORES["azul"]])
    ax.set_title(metrica)
    ax.bar_label(barras, fmt="%.3f", padding=3)
    ax.set_ylim(0, max(valores) * 1.25)
axes[1, 1].bar(["MAE", "RMSE"], [mejora_mae, mejora_rmse], color=[COLORES["verde"], COLORES["morado"]])
axes[1, 1].set_title("Reducción relativa del error")
axes[1, 1].set_ylabel("Mejora (%)")
axes[1, 1].set_ylim(0, 40)
fig.suptitle("Comparación final en el conjunto de prueba", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Reducción de MAE: {mejora_mae:.2f}%")
print(f"Reducción de RMSE: {mejora_rmse:.2f}%")


## 8.16 Ejemplos de pronóstico


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
indices_muestra = [0, 100, 500, 1000]

for ax, idx in zip(axes.flatten(), indices_muestra):
    ax.plot(range(Tout), y_test_real[idx], label="Real", marker="o")
    ax.plot(range(Tout), y_pred_baseline_real[idx], label="Baseline", marker="x", linestyle="--")
    ax.plot(range(Tout), y_pred_seq2seq_real[idx], label="Seq2Seq+Attention", marker="s")
    ax.set_title(f"Muestra {idx}")
    ax.set_xlabel("Horizonte (h)")
    ax.set_ylabel("T (°C)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


El mejor modelo aparece en la **época 8**, tiene **57.058 parámetros** y obtiene **MAE = 1,7002 °C, RMSE = 2,2931 °C y R² = 0,9132**. Reduce el MAE del baseline aproximadamente **31,56 %** y el RMSE **29,04 %**.

El Encoder lee las 48 horas, Attention decide qué posiciones consultar y el Decoder genera las 24 temperaturas. En prueba, el Seq2Seq supera claramente al patrón diario, por lo que aprendió información adicional y no solo copió el día anterior.


# PARTE 9. Análisis de Attention
- Se extrajo una matriz por muestra y un promedio sobre prueba.
- Se visualizaron mapas de calor.
- Se comparó la posición de máxima atención con la referencia diaria correcta, que es diagonal.
- Se respondieron las cinco preguntas del taller con una interpretación prudente.

primero indique el objetivo; después explique la decisión técnica; finalmente muestre las cifras o gráficas y cierre con la conclusión breve.


## 9.1 Matriz individual


In [ ]:
# attention_test shape: (N, Tout, Tin) = (N, 24, 48)
idx_muestra = 0
A_muestra = attention_test[idx_muestra]  # shape (24, 48)

print("Matriz de atención de la muestra:", A_muestra.shape)
print("Filas (24) = horas futuras predichas")
print("Columnas (48) = horas pasadas observadas")

## 9.2 Mapa de calor de una muestra


In [ ]:
plt.figure(figsize=(9, 8))
sns.heatmap(A_muestra, cmap="mako", xticklabels=4, yticklabels=2, cbar_kws={"label": "Peso de Attention"})
plt.title(f"Attention de la muestra {idx_muestra}: 48 horas -> 24 horizontes")
plt.xlabel("Índice de entrada (0=t-47; 47=t)")
plt.ylabel("Horizonte futuro (t+1 a t+24)")
plt.tight_layout()
plt.show()


## 9.3 Attention promedio


In [ ]:
A_promedio = attention_test.mean(axis=0)
plt.figure(figsize=(9, 8))
sns.heatmap(A_promedio, cmap="mako", xticklabels=4, yticklabels=2, cbar_kws={"label": "Peso promedio"})
plt.title("Attention promedio en prueba")
plt.xlabel("Índice de entrada (0=t-47; 47=t)")
plt.ylabel("Horizonte futuro (t+1 a t+24)")
plt.tight_layout()
plt.show()


## 9.4 Comparación correcta con el patrón diario

La hora equivalente del día anterior no ocupa una columna fija. Para el horizonte `h=1` corresponde al índice 24 y para `h=24` al índice 47; por eso la referencia correcta es una **diagonal de 24 a 47**, no una línea horizontal.


In [ ]:
posicion_max_atencion = A_promedio.argmax(axis=1)
horizontes = np.arange(1, Tout + 1)
referencia_diaria = horizontes + 23  # índices 24..47 para h=1..24

plt.figure(figsize=(8, 8))
plt.plot(horizontes, posicion_max_atencion, marker="o", linewidth=2.2,
         color=COLORES["morado"], label="Máximo observado")
plt.plot(horizontes, referencia_diaria, linestyle="--", linewidth=2,
         color=COLORES["naranja"], label="Referencia diaria esperada")
plt.title("Máximo de Attention vs. referencia diaria")
plt.xlabel("Horizonte futuro h (1 a 24)")
plt.ylabel("Índice de la hora pasada (0=t-47; 47=t)")
plt.ylim(0, 49)
plt.legend()
plt.tight_layout()
plt.show()
print("Posiciones de máxima atención:")
print(posicion_max_atencion)


## 9.5 Perfiles para horizontes cercanos y lejanos


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 9), sharex=True)
axes[0].plot(range(Tin), A_promedio[0], label="t+1", color=COLORES["azul"])
axes[0].plot(range(Tin), A_promedio[5], label="t+6", color=COLORES["verde"])
axes[0].set_title("Attention en horizontes cercanos")
axes[0].set_ylabel("Peso")
axes[0].legend()
axes[1].plot(range(Tin), A_promedio[12], label="t+13", color=COLORES["naranja"])
axes[1].plot(range(Tin), A_promedio[23], label="t+24", color=COLORES["morado"])
axes[1].set_title("Attention en horizontes lejanos")
axes[1].set_xlabel("Índice de la hora pasada")
axes[1].set_ylabel("Peso")
axes[1].legend()
plt.tight_layout()
plt.show()


## 9.6 Respuestas a las cinco preguntas del taller

**1. ¿Qué representan las filas?**  
Cada fila representa uno de los 24 horizontes futuros: desde `t+1` hasta `t+24`. La fila contiene la distribución de pesos usada por el decoder para ese horizonte.

**2. ¿Qué representan las columnas?**  
Cada columna representa una de las 48 horas de entrada. Con índices de 0 a 47, la columna 0 corresponde a `t-47` y la 47 corresponde a la última observación `t`.

**3. ¿Qué significa un valor elevado de Attention?**  
Significa que esa posición recibió un peso mayor al construir el vector de contexto para una predicción. Es una señal interna de foco temporal del modelo, pero **no debe interpretarse como causalidad ni como importancia global definitiva**.

**4. ¿Las diferentes horas futuras atienden las mismas regiones?**  
No exactamente. Los máximos cambian con el horizonte y los perfiles cercanos y lejanos distribuyen sus pesos de manera diferente. Esto indica que el decoder adapta el contexto temporal a cada salida.

**5. ¿Se observa algún patrón aproximadamente diario?**  
Se observa una estructura temporal y varios máximos se desplazan de forma ordenada, pero la comparación con la diagonal diaria esperada muestra que el patrón **no es perfecto ni uniforme**. Por tanto, la evidencia es parcial: Attention complementa la periodicidad diaria, pero esta gráfica por sí sola no demuestra que el modelo copie exactamente el día anterior.


### Conclusión breve

La matriz tiene la forma exigida **24 x 48**. El foco cambia entre horizontes y no sigue de manera perfecta la diagonal del día anterior. Attention aporta una interpretación útil del contexto consultado, pero sus pesos deben leerse con cautela y no como relaciones causales.

Las filas son las 24 predicciones y las columnas son las 48 horas observadas. Un color intenso indica mayor peso para esa salida. El patrón cambia según el horizonte y solo coincide parcialmente con la referencia diaria, lo que sugiere un uso más flexible del pasado.


# PARTE 12 (XII). Comparación de modelos

## Propósito

Comparar formalmente el baseline de periodicidad diaria y el modelo Seq2Seq con Bahdanau Attention utilizando:

- número de parámetros;
- MAE;
- RMSE;
- R²;
- tiempo de entrenamiento.


In [ ]:
# ============================================================
# PARTE 12: TABLA Y GRÁFICAS COMPARATIVAS
# ============================================================

from IPython.display import display, Markdown

# El baseline no tiene parámetros entrenables ni necesita entrenamiento.
parametros_baseline = 0
parametros_seq2seq = model_seq2seq.count_params()

# Si se ejecutó la celda de entrenamiento, esta variable contiene
# el tiempo total medido. Se deja una alternativa segura.
tiempo_seq2seq_seg = globals().get("training_time_seq2seq", np.nan)

if np.isfinite(tiempo_seq2seq_seg):
    tiempo_seq2seq_texto = (
        f"{tiempo_seq2seq_seg:.2f} s "
        f"({tiempo_seq2seq_seg / 60:.2f} min)"
    )
else:
    tiempo_seq2seq_texto = "No registrado"

tabla_comparacion = pd.DataFrame({
    "Modelo": [
        "Baseline diario",
        "Seq2Seq + Attention"
    ],
    "Parámetros": [
        parametros_baseline,
        parametros_seq2seq
    ],
    "MAE (°C)": [
        metricas_baseline["MAE"],
        metricas_seq2seq["MAE"]
    ],
    "RMSE (°C)": [
        metricas_baseline["RMSE"],
        metricas_seq2seq["RMSE"]
    ],
    "R²": [
        metricas_baseline["R2"],
        metricas_seq2seq["R2"]
    ],
    "Tiempo de entrenamiento": [
        "No requiere",
        tiempo_seq2seq_texto
    ]
})

print("Comparación final de modelos:")
display(
    tabla_comparacion.style
    .format({
        "Parámetros": "{:,.0f}",
        "MAE (°C)": "{:.4f}",
        "RMSE (°C)": "{:.4f}",
        "R²": "{:.4f}"
    })
    .background_gradient(
        subset=["MAE (°C)", "RMSE (°C)"],
        cmap="Oranges_r"
    )
    .background_gradient(
        subset=["R²"],
        cmap="Blues"
    )
)

# Mejoras relativas del Seq2Seq frente al baseline.
mejora_mae = (
    100
    * (metricas_baseline["MAE"] - metricas_seq2seq["MAE"])
    / metricas_baseline["MAE"]
)

mejora_rmse = (
    100
    * (metricas_baseline["RMSE"] - metricas_seq2seq["RMSE"])
    / metricas_baseline["RMSE"]
)

mejora_r2 = (
    metricas_seq2seq["R2"] - metricas_baseline["R2"]
)

# Gráfica comparativa cuadrada.
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

configuracion = [
    ("MAE", "MAE (°C)", "MAE: menor es mejor"),
    ("RMSE", "RMSE (°C)", "RMSE: menor es mejor"),
    ("R2", "R²", "R²: mayor es mejor")
]

colores_modelos = ["#F97316", "#2563EB"]

for ax, (clave, etiqueta, titulo) in zip(
    axes.flat[:3],
    configuracion
):
    valores = [
        metricas_baseline[clave],
        metricas_seq2seq[clave]
    ]

    barras = ax.bar(
        ["Baseline", "Seq2Seq"],
        valores,
        color=colores_modelos,
        width=0.65
    )

    ax.set_title(titulo, fontweight="bold")
    ax.set_ylabel(etiqueta)
    ax.bar_label(barras, fmt="%.4f", padding=4)
    ax.set_ylim(0, max(valores) * 1.25)

# Cuarta gráfica: mejoras porcentuales.
barras = axes[1, 1].bar(
    ["Reducción\nMAE", "Reducción\nRMSE"],
    [mejora_mae, mejora_rmse],
    color=["#10B981", "#8B5CF6"],
    width=0.65
)

axes[1, 1].set_title(
    "Mejora relativa del Seq2Seq",
    fontweight="bold"
)
axes[1, 1].set_ylabel("Mejora (%)")
axes[1, 1].set_ylim(
    0,
    max(mejora_mae, mejora_rmse) * 1.30
)
axes[1, 1].bar_label(
    barras,
    labels=[
        f"{mejora_mae:.2f}%",
        f"{mejora_rmse:.2f}%"
    ],
    padding=4
)

fig.suptitle(
    "Baseline diario vs. Seq2Seq + Attention",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

display(Markdown(
    f"""
### Conclusión breve

El modelo **Seq2Seq + Attention** supera al baseline diario:

- reduce el MAE en **{mejora_mae:.2f} %**;
- reduce el RMSE en **{mejora_rmse:.2f} %**;
- incrementa el R² en **{mejora_r2:.4f}**;
- utiliza **{parametros_seq2seq:,} parámetros entrenables**.

El aumento de complejidad y tiempo de entrenamiento se justifica porque el
modelo neuronal obtiene predicciones considerablemente más precisas.
"""
))

 En esta tabla comparamos el modelo neuronal contra la referencia mínima del taller. El baseline no requiere entrenamiento ni tiene parámetros, pero presenta mayor error. El Seq2Seq tiene 57.058 parámetros y requiere más tiempo computacional; a cambio, reduce aproximadamente 31,6 % el MAE y 29 % el RMSE, además de aumentar el R² de 0,8276 a 0,9132.

# PARTE 13 (XIII). Error por horizonte

Analizar cómo cambia el RMSE desde la primera hasta la vigesimocuarta hora futura.

La métrica global resume todo el pronóstico en un único valor, pero el RMSE por horizonte permite determinar si el modelo pierde precisión a medida que intenta predecir horas más lejanas.

 Debido al alcance de la actividad, la gráfica compara únicamente el baseline y Seq2Seq + Attention. El Transformer se excluye por indicación.

In [ ]:
# ============================================================
# PARTE 13: RMSE PARA CADA UNO DE LOS 24 HORIZONTES
# ============================================================

from IPython.display import display, Markdown

def asegurar_formato_2d(array, nombre):
    """
    Convierte (N, 24, 1) en (N, 24) cuando sea necesario
    y verifica que existan exactamente 24 horizontes.
    """
    array = np.asarray(array)

    if array.ndim == 3 and array.shape[-1] == 1:
        array = array[..., 0]

    if array.ndim != 2:
        raise ValueError(
            f"{nombre} debe tener forma (N, 24) o (N, 24, 1). "
            f"Forma recibida: {array.shape}"
        )

    if array.shape[1] != 24:
        raise ValueError(
            f"{nombre} debe contener 24 horizontes. "
            f"Forma recibida: {array.shape}"
        )

    return array


# Aseguramos que todas las matrices tengan forma (N, 24).
y_real_h = asegurar_formato_2d(
    y_test_real,
    "y_test_real"
)

y_baseline_h = asegurar_formato_2d(
    y_pred_baseline_real,
    "y_pred_baseline_real"
)

y_seq2seq_h = asegurar_formato_2d(
    y_pred_seq2seq_real,
    "y_pred_seq2seq_real"
)

# Verificación de compatibilidad.
if not (
    y_real_h.shape == y_baseline_h.shape == y_seq2seq_h.shape
):
    raise ValueError(
        "Los valores reales y las predicciones deben tener la misma forma. "
        f"Real: {y_real_h.shape}, "
        f"Baseline: {y_baseline_h.shape}, "
        f"Seq2Seq: {y_seq2seq_h.shape}"
    )

# RMSE independiente para cada horizonte.
rmse_horizonte_baseline = np.sqrt(
    np.mean(
        np.square(y_real_h - y_baseline_h),
        axis=0
    )
)

rmse_horizonte_seq2seq = np.sqrt(
    np.mean(
        np.square(y_real_h - y_seq2seq_h),
        axis=0
    )
)

horizontes = np.arange(1, 25)

tabla_horizontes = pd.DataFrame({
    "Horizonte (h)": horizontes,
    "RMSE Baseline (°C)": rmse_horizonte_baseline,
    "RMSE Seq2Seq (°C)": rmse_horizonte_seq2seq,
    "Reducción del RMSE (°C)": (
        rmse_horizonte_baseline
        - rmse_horizonte_seq2seq
    )
})

display(
    tabla_horizontes.style
    .format({
        "RMSE Baseline (°C)": "{:.4f}",
        "RMSE Seq2Seq (°C)": "{:.4f}",
        "Reducción del RMSE (°C)": "{:.4f}"
    })
    .background_gradient(
        subset=["RMSE Baseline (°C)"],
        cmap="Oranges"
    )
    .background_gradient(
        subset=["RMSE Seq2Seq (°C)"],
        cmap="Blues"
    )
)

# Gráfica comparativa.
fig, ax = plt.subplots(figsize=(10, 9))

ax.plot(
    horizontes,
    rmse_horizonte_baseline,
    marker="o",
    markersize=6,
    linewidth=2.5,
    color="#F97316",
    label="Baseline diario"
)

ax.plot(
    horizontes,
    rmse_horizonte_seq2seq,
    marker="s",
    markersize=6,
    linewidth=2.5,
    color="#2563EB",
    label="Seq2Seq + Attention"
)

# Sombreado de la diferencia entre los modelos.
ax.fill_between(
    horizontes,
    rmse_horizonte_baseline,
    rmse_horizonte_seq2seq,
    where=(
        rmse_horizonte_seq2seq
        <= rmse_horizonte_baseline
    ),
    color="#10B981",
    alpha=0.15,
    label="Ventaja del Seq2Seq"
)

ax.set_title(
    "RMSE por horizonte de predicción",
    fontsize=16,
    fontweight="bold"
)
ax.set_xlabel("Horizonte futuro (horas)")
ax.set_ylabel("RMSE (°C)")
ax.set_xticks(horizontes)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

# Resumen automático.
horizontes_mejorados = int(
    np.sum(
        rmse_horizonte_seq2seq
        < rmse_horizonte_baseline
    )
)

rmse_promedio_baseline = float(
    np.mean(rmse_horizonte_baseline)
)

rmse_promedio_seq2seq = float(
    np.mean(rmse_horizonte_seq2seq)
)

mejor_horizonte_seq2seq = int(
    np.argmin(rmse_horizonte_seq2seq) + 1
)

peor_horizonte_seq2seq = int(
    np.argmax(rmse_horizonte_seq2seq) + 1
)

display(Markdown(
    f"""
### Conclusión breve

- El Seq2Seq obtiene menor RMSE que el baseline en
  **{horizontes_mejorados} de 24 horizontes**.
- El promedio de los RMSE por horizonte es
  **{rmse_promedio_baseline:.4f} °C** para el baseline y
  **{rmse_promedio_seq2seq:.4f} °C** para Seq2Seq.
- El menor error del Seq2Seq aparece en el horizonte
  **h = {mejor_horizonte_seq2seq}**.
- El mayor error aparece en el horizonte
  **h = {peor_horizonte_seq2seq}**.

La gráfica permite comprobar si la incertidumbre aumenta con la distancia
temporal y si la ventaja del Seq2Seq se mantiene a lo largo del día futuro.
"""
))

 En esta gráfica el eje horizontal representa las 24 horas futuras y el eje vertical muestra el RMSE de cada una. Una curva más baja significa mejor precisión. Esta evaluación es más informativa que una sola métrica global porque permite observar cómo cambia el desempeño cuando aumenta la distancia del pronóstico. El área sombreada indica los horizontes en los que Seq2Seq supera al baseline.